# Demos: Appendix A: Reading Other Sources

Each cell below is one demo from the slides; the heading matches the pill on the slide. Run the first cell, then run the demos in any order with Shift+Enter. The same code is in the `.py` files in this folder.

In [ ]:
# Run this cell first: the demos read their data files from the working folder.
import os
from pathlib import Path

COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir())
os.chdir(COURSE / "datasets/demos/A-reading-other-sources")
print("Working folder:", Path.cwd())

## demo01 · Several files

In [ ]:
from glob import glob

import pandas as pd

filenames = sorted(glob("sales-*-2026.csv"))
print(filenames)
frames = [pd.read_csv(name, parse_dates=["Date"]).assign(source=name) for name in filenames]
sales = pd.concat(frames, ignore_index=True)
print(sales["source"].value_counts().sort_index())
print(len(sales), sales["Date"].min(), sales["Date"].max())

## demo02 · Excel

In [ ]:
import pandas as pd

march = pd.read_excel("shop.xlsx", sheet_name="March", skiprows=3)
print(march.head(3))

sheets = pd.read_excel("shop.xlsx", sheet_name=None)
print(list(sheets))
stores = pd.read_excel("shop.xlsx", sheet_name="Stores")
print(stores)

## demo03 · JSON

In [ ]:
import json

import pandas as pd

flat = pd.read_json("orders-flat.json")
print(flat)

with open("orders.json") as file:
    records = json.load(file)
orders = pd.json_normalize(records)
print(orders.columns.tolist())
print(orders[["order_id", "customer.city"]])
items = pd.json_normalize(records, record_path="items", meta=["order_id"])
print(items)

## demo04 · SQL

In [ ]:
import sqlite3

import pandas as pd

con = sqlite3.connect(":memory:")
sales = pd.read_csv("sales-feb-2026.csv")
sales.to_sql("sales", con, index=False)

query = """
SELECT Company, SUM(Units) AS units, COUNT(*) AS orders
FROM sales
GROUP BY Company
ORDER BY units DESC
"""
print(pd.read_sql(query, con))
print(pd.read_sql("SELECT * FROM sales WHERE Units > ?", con, params=(15,)))
con.close()